# Machine Learning 2025W — Exercise 1  
### Dataset Description and Exploration (breast cancer)

**Group Members:**  
- Full Name 1  
- Full Name 2  
- Full Name 3  

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
import ssl
import certifi
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## Load data

In [ ]:
file_path = '../datasets/184-702-tu-ml-2025w-breast-cancer/'
df= pd.read_csv(file_path+ "breast-cancer-diagnostic.shuf.lrn.csv")
df_kaggle_test= pd.read_csv(file_path+ "breast-cancer-diagnostic.shuf.tes.csv")
df=df.set_index("ID")
df.head()


In [ ]:
df.shape

### Basic Information

In [ ]:
df.info()
df.describe()
df.isna().sum()

In [ ]:
df_kaggle_test.info()
df_kaggle_test.describe()
df_kaggle_test.isna().sum()

### Target Variable Analysis

In [ ]:
target_col = 'class' 
sns.countplot(x=target_col, data=df)
plt.title('Breast cancer class - Training set')
plt.xlabel('Breast cancer class')
plt.show()

### Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(24, 16))
plt.suptitle("Numeric Feature Distributions")
plt.show()

# There are no categorical variables

# Preprocessing


## Target varaiable 

Since most models require numerical input, we will map the class to 0(no cancer) and 1(cancer).

In [ ]:
def target_to_num(x):
    if x==False:
        return 0
    else:
        return 1
    
df["class"]= df["class"].apply(target_to_num)

sns.countplot(x=target_col, data=df)
plt.title('Breast cancer class - Training set')
plt.xlabel('Breast cancer class')
plt.show()

## Input variables

### Check for outliers

In [ ]:
# Select numeric columns (excluding target variable)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'class' in numeric_cols:
    numeric_cols.remove('class') # class not included

print(f"Number of numeric columns to check: {len(numeric_cols)}")

# Define and run outlier detection function
def detect_outliers_iqr(df, columns):
    
    outlier_info = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_percentage = (outlier_count / len(df)) * 100
        
        outlier_info[col] = {
            'count': outlier_count,
            'percentage': outlier_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'min': df[col].min(),
            'max': df[col].max(),
            'mean': df[col].mean(),
            'std': df[col].std()
        }
    
    return pd.DataFrame(outlier_info).T.sort_values('percentage', ascending=False)

# Run outlier detection
outlier_summary = detect_outliers_iqr(df, numeric_cols)
print("\nOutlier Summary:")
print(outlier_summary.head(10))

# Create the boxplots
n_cols = 5
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    df.boxplot(column=col, ax=axes[idx])
    outlier_pct = outlier_summary.loc[col, 'percentage']
    axes[idx].set_title(f'{col}', fontsize=9)
    axes[idx].tick_params(labelsize=8)


plt.tight_layout()
plt.show()

To decide how to handle the outliers, we first need to consider the background of the data. In this case it is medical data, where outliers can be very important for the diagnosis. Therefore, we will look if the outliers are data errors or just extreme values that should be kept in the dataset. 
In this case we flag data points as outlier if they are lower then Q1 - 1.5 * IQR or higher then Q3 + 1.5 * IQR.
In this data, the columns : areaStdErr, smoothnessStdErr, radiusStdErr, perimeterStdErr, fractalDimensionWorst, concavityStdErr, symmetryStdErr, areaMean, areaWorst, and symmetryWorst show outliers with a percentage of 11%-3 %.
We further looked at the plots and the min and max values of these columns and found that the outliers are not data errors but extreme values. Therefore, we will keep them in the dataset.

### Handle missing values

In [ ]:
print(df.isnull().sum())
print((df.astype(str)=="?").any())
print((df.astype(str)=="nan").any())

WE can see that there are no missing values in the dataset. This also alignes with the discriotion of the datatset provided.

In [ ]:
print(df_kaggle_test.isnull().sum())
print((df_kaggle_test.astype(str)=="?").any())
print((df_kaggle_test.astype(str)=="nan").any())

Also no missing values in  the test dataset.

## Split training data into train and test sets

In [ ]:
X = df.drop("class", axis=1)  # features
y = df["class"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% of data for testing
    random_state=42,     # ensures reproducibility
    stratify=y           # keeps same class proportions in train/test
)

X_train.head()

## Scale data

In [ ]:
#Scale training and test data
X_train_scaled=X_train.copy()
X_test_scaled= X_test.copy()
scaler = StandardScaler()
X_train_scaled[numeric_cols]= scaler.fit_transform(X_train_scaled[numeric_cols])
X_test_scaled[numeric_cols]= scaler.fit_transform(X_test_scaled[numeric_cols])
X_test_scaled.describe()

In [ ]:
print(numeric_cols)